# 07 从零采集 MuJoCo LeRobot 示教数据

        这一节把 `5.language_env.ipynb` 的交互采集流程整理成可配置、可复核的版本。最终产物是一个包含双相机图像、6 维关节状态、7 维动作和红/蓝杯语言指令的 LeRobot 数据集。

        采集不是纯 GPU 任务，NVIDIA 或 AMD 设备都可以完成。真正需要保持一致的是 MuJoCo 场景、LeRobot 数据格式、控制频率、state/action 定义和成功判定。远端 Jupyter 无法稳定接收 MuJoCo viewer 键盘事件时，可以在有桌面的机器采集，再把数据目录同步到 AMD 训练机。


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys


def find_topic_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "assets" / "metrics_snapshot.json").exists():
            return candidate
    raise RuntimeError("请从 AMD ROCm 专题目录或 notebooks 子目录启动 Jupyter。")


TOPIC_ROOT = find_topic_root()
ASSET_DIR = TOPIC_ROOT / "assets"
NOTEBOOK_DIR = TOPIC_ROOT / "notebooks"
PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", "/path/to/every-embodied/mujoco_pnp"))
DATA_ROOT = Path(os.environ.get("DATA_ROOT", "/path/to/datasets/every_embodied"))
OUTPUT_ROOT = Path(os.environ.get("OUTPUT_ROOT", TOPIC_ROOT / "outputs"))
MODEL_ROOT = Path(os.environ.get("MODEL_ROOT", PROJECT_ROOT / "ckpt"))

print("TOPIC_ROOT =", TOPIC_ROOT)
print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_ROOT =", DATA_ROOT)
print("OUTPUT_ROOT =", OUTPUT_ROOT)
print("MODEL_ROOT =", MODEL_ROOT)


In [ ]:
try:
    from IPython.display import Image, Markdown, display
except Exception:
    class Markdown(str):
        pass

    def display(obj):
        print(obj)

    def Image(filename=None, width=None):
        return f"[image] {filename}"


def show_asset(filename, width=960):
    path = ASSET_DIR / filename
    if path.exists():
        display(Image(filename=str(path), width=width))
    else:
        print(f"缺少素材：{path}")


def md_table(headers, rows):
    lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]
    for row in rows:
        lines.append("| " + " | ".join(str(x) for x in row) + " |")
    display(Markdown("\n".join(lines)))


In [ ]:
import shlex

try:
    import yaml
except ImportError as exc:
    raise RuntimeError("当前环境缺少 PyYAML，请先执行 pip install pyyaml。") from exc


def require_project_layout():
    required = [
        PROJECT_ROOT / "train_model.py",
        PROJECT_ROOT / "env_config.py",
        PROJECT_ROOT / "asset" / "example_scene_y2.xml",
        PROJECT_ROOT / "mujoco_env" / "y_env2.py",
    ]
    missing = [path for path in required if not path.exists()]
    if missing:
        raise FileNotFoundError(
            "PROJECT_ROOT 不是 04mujoco 教程目录，缺少：\n"
            + "\n".join(str(path) for path in missing)
        )
    return True


def dataset_report(dataset_root):
    dataset_root = Path(dataset_root)
    info_path = dataset_root / "meta" / "info.json"
    tasks_path = dataset_root / "meta" / "tasks.jsonl"
    if not info_path.exists():
        raise FileNotFoundError(f"找不到 LeRobot 元数据：{info_path}")
    info = json.loads(info_path.read_text(encoding="utf-8"))
    tasks = []
    if tasks_path.exists():
        tasks = [
            json.loads(line)["task"]
            for line in tasks_path.read_text(encoding="utf-8").splitlines()
            if line.strip()
        ]
    features = info.get("features", {})
    rows = [
        ("repo_id", info.get("repo_id", "")),
        ("episodes", info.get("total_episodes", 0)),
        ("frames", info.get("total_frames", 0)),
        ("fps", info.get("fps", "")),
        ("state shape", features.get("observation.state", {}).get("shape")),
        ("action shape", features.get("action", {}).get("shape")),
        ("tasks", " / ".join(tasks)),
    ]
    md_table(["数据项", "读取结果"], rows)
    return info, tasks


def make_training_config(
    policy_type,
    dataset_repo_id,
    dataset_root,
    output_dir,
    steps,
    batch_size,
    chunk_size,
    n_action_steps,
    save_freq,
    seed=42,
):
    return {
        "dataset": {"repo_id": dataset_repo_id, "root": str(Path(dataset_root))},
        "policy": {
            "type": policy_type,
            "chunk_size": int(chunk_size),
            "n_action_steps": int(n_action_steps),
            "device": "cuda",
        },
        "save_checkpoint": True,
        "output_dir": str(Path(output_dir)),
        "batch_size": int(batch_size),
        "job_name": Path(output_dir).name,
        "resume": False,
        "seed": int(seed),
        "num_workers": 4,
        "steps": int(steps),
        "eval_freq": 0,
        "log_freq": max(1, min(50, int(steps))),
        "save_freq": int(save_freq),
        "use_policy_training_preset": True,
        "wandb": {
            "enable": False,
            "project": f"every_embodied_{policy_type}",
            "entity": None,
            "disable_artifact": True,
        },
    }


def write_training_config(name, config):
    config_dir = OUTPUT_ROOT / "configs"
    config_dir.mkdir(parents=True, exist_ok=True)
    path = config_dir / f"{name}.yaml"
    path.write_text(
        yaml.safe_dump(config, allow_unicode=True, sort_keys=False),
        encoding="utf-8",
    )
    print("已写出配置：", path)
    return path


def training_command(config_path):
    return [
        sys.executable,
        str(PROJECT_ROOT / "train_model.py"),
        "--config_path",
        str(Path(config_path)),
    ]


def run_training(config_path, enabled=False):
    require_project_layout()
    command = training_command(config_path)
    print("$", shlex.join(command))
    if not enabled:
        print("当前只预览命令。确认配置后，把对应 RUN_* 开关改为 True。")
        return None
    env = os.environ.copy()
    env["PYTHONPATH"] = f"{PROJECT_ROOT}:{env.get('PYTHONPATH', '')}"
    return subprocess.run(command, cwd=PROJECT_ROOT, env=env, check=True)


def show_rocm_resources():
    command = ["rocm-smi", "--showuse", "--showmemuse", "--showtemp"]
    print("$", shlex.join(command))
    if shutil.which(command[0]) is None:
        print("未找到 rocm-smi。确认当前机器是否安装 ROCm。")
        return
    subprocess.run(command, check=False)


## Checkpoint 1：设置采集目录与安全开关


In [ ]:
DATASET_REPO_ID = os.environ.get("DATASET_REPO_ID", "datawhale_eai_pnp_language_local")
COLLECTION_ROOT = Path(
    os.environ.get("COLLECTION_ROOT", DATA_ROOT / "omy_pnp_language")
).expanduser()
NUM_DEMOS = int(os.environ.get("NUM_DEMOS", "20"))
SEED_START = int(os.environ.get("SEED_START", "0"))
INSTRUCTIONS = [
    "Place the red mug on the plate.",
    "Place the blue mug on the plate.",
]

RUN_INTERACTIVE_COLLECTION = False
OVERWRITE_DATASET = False

print("dataset repo_id =", DATASET_REPO_ID)
print("collection root =", COLLECTION_ROOT)
print("num demos =", NUM_DEMOS)
print("instructions =", INSTRUCTIONS)


`COLLECTION_ROOT` 应放在大容量数据盘。默认不开启采集，也不会删除已有目录；只有明确把 `OVERWRITE_DATASET=True` 后才允许覆盖。20 条可以跑通语言条件小实验，想做位置泛化时建议采 30–50 条高质量轨迹，并让红杯、蓝杯和初始位置都得到覆盖。


## Checkpoint 2：检查项目、显示和数据 schema


In [ ]:
require_project_layout()
print("MuJoCo scene =", PROJECT_ROOT / "asset" / "example_scene_y2.xml")
print("DISPLAY =", os.environ.get("DISPLAY"))
if not os.environ.get("DISPLAY"):
    print("当前没有 DISPLAY。交互采集需要本地桌面、远程桌面或可用的 X11 会话。")

FEATURES = {
    "observation.image": {
        "dtype": "image",
        "shape": (256, 256, 3),
        "names": ["height", "width", "channels"],
    },
    "observation.wrist_image": {
        "dtype": "image",
        "shape": (256, 256, 3),
        "names": ["height", "width", "channels"],
    },
    "observation.state": {
        "dtype": "float32",
        "shape": (6,),
        "names": ["state"],
    },
    "action": {
        "dtype": "float32",
        "shape": (7,),
        "names": ["action"],
    },
    "obj_init": {
        "dtype": "float32",
        "shape": (9,),
        "names": [
            "red_x", "red_y", "red_z",
            "blue_x", "blue_y", "blue_z",
            "plate_x", "plate_y", "plate_z",
        ],
    },
}
md_table(
    ["字段", "shape", "作用"],
    [
        ("observation.image", "256x256x3", "固定相机"),
        ("observation.wrist_image", "256x256x3", "腕部相机"),
        ("observation.state", "6", "当前 6 关节状态"),
        ("action", "7", "下一步 6 关节目标 + 夹爪"),
        ("obj_init", "9", "三件物体初始 xyz，仅用于回放/审计"),
    ],
)


## Checkpoint 3：定义严格成功判定


In [ ]:
import numpy as np


def collection_success(env, initial_target_z, max_target_lift, max_lifted_run):
    target_pos = np.asarray(env.env.get_p_body(env.obj_target), dtype=np.float64)
    plate_pos = np.asarray(env.env.get_p_body("body_obj_plate_11"), dtype=np.float64)
    target_R = np.asarray(env.env.get_R_body(env.obj_target), dtype=np.float64)
    legacy_success = bool(env.check_success())
    upright_cos = float(target_R[2, 2])
    physical_success = bool(
        legacy_success
        and max_target_lift >= 0.03
        and max_lifted_run >= 3
        and upright_cos >= 0.7
        and abs(float(target_pos[2] - plate_pos[2])) < 0.15
    )
    return {
        "legacy_success": legacy_success,
        "physical_success": physical_success,
        "xy_dist": float(np.linalg.norm(target_pos[:2] - plate_pos[:2])),
        "max_target_lift": float(max_target_lift),
        "max_lifted_run": int(max_lifted_run),
        "upright_cos": upright_cos,
    }


旧 `check_success()` 只看终态几何关系，杯子被推到盘子附近也可能触发。这里额外要求杯子至少抬升 3 cm、连续保持 3 个控制步、终态保持直立，并且杯子高度与盘子相符。只有 `physical_success=True` 才写入 episode。


## Checkpoint 4：创建或加载 LeRobot 数据集


In [ ]:
def create_or_load_dataset():
    from lerobot.common.datasets.lerobot_dataset import LeRobotDataset

    COLLECTION_ROOT.parent.mkdir(parents=True, exist_ok=True)
    if COLLECTION_ROOT.exists() and OVERWRITE_DATASET:
        shutil.rmtree(COLLECTION_ROOT)
    if COLLECTION_ROOT.exists():
        print("继续写入已有数据集：", COLLECTION_ROOT)
        return LeRobotDataset(DATASET_REPO_ID, root=COLLECTION_ROOT)
    print("创建新数据集：", COLLECTION_ROOT)
    return LeRobotDataset.create(
        repo_id=DATASET_REPO_ID,
        root=COLLECTION_ROOT,
        robot_type="omy",
        fps=20,
        features=FEATURES,
        image_writer_threads=10,
        image_writer_processes=0,
    )


## Checkpoint 5：键盘采集完整循环


In [ ]:
def collect_demonstrations():
    import random
    import numpy as np
    from PIL import Image
    from mujoco_env.y_env2 import SimpleEnv2

    dataset = create_or_load_dataset()
    xml_path = PROJECT_ROOT / "asset" / "example_scene_y2.xml"

    def reset_episode(env, episode_id):
        seed = SEED_START + episode_id
        np.random.seed(seed)
        random.seed(seed)
        env.reset(seed=None)
        instruction = INSTRUCTIONS[episode_id % len(INSTRUCTIONS)]
        env.set_instruction(instruction)
        initial_z = float(env.env.get_p_body(env.obj_target)[2])
        print(f"episode={episode_id} seed={seed} task={instruction}")
        return initial_z

    np.random.seed(SEED_START)
    random.seed(SEED_START)
    env = SimpleEnv2(str(xml_path), seed=None, state_type="joint_angle")
    episode_id = 0
    record_flag = False
    initial_target_z = reset_episode(env, episode_id)
    max_target_lift = 0.0
    lifted_run = 0
    max_lifted_run = 0

    try:
        while env.env.is_viewer_alive() and episode_id < NUM_DEMOS:
            env.step_env()
            if not env.env.loop_every(HZ=20):
                continue

            target_z = float(env.env.get_p_body(env.obj_target)[2])
            lift = target_z - initial_target_z
            max_target_lift = max(max_target_lift, lift)
            lifted_run = lifted_run + 1 if lift >= 0.03 else 0
            max_lifted_run = max(max_lifted_run, lifted_run)
            status = collection_success(env, initial_target_z, max_target_lift, max_lifted_run)

            if record_flag and status["physical_success"]:
                dataset.save_episode()
                print("saved:", json.dumps(status, ensure_ascii=False))
                episode_id += 1
                record_flag = False
                if episode_id >= NUM_DEMOS:
                    break
                initial_target_z = reset_episode(env, episode_id)
                max_target_lift = 0.0
                lifted_run = 0
                max_lifted_run = 0
                continue

            teleop_delta, reset = env.teleop_robot()
            if reset:
                dataset.clear_episode_buffer()
                record_flag = False
                initial_target_z = reset_episode(env, episode_id)
                max_target_lift = 0.0
                lifted_run = 0
                max_lifted_run = 0
                continue

            if not record_flag and np.any(np.abs(teleop_delta) > 1e-8):
                record_flag = True
                print("Start recording")

            agent_image, wrist_image = env.grab_image()
            state = env.get_joint_state()[:6].astype(np.float32)
            env.step(teleop_delta)
            target_action = env.q[:7].astype(np.float32)

            if record_flag:
                dataset.add_frame(
                    {
                        "observation.image": np.asarray(Image.fromarray(agent_image).resize((256, 256))),
                        "observation.wrist_image": np.asarray(Image.fromarray(wrist_image).resize((256, 256))),
                        "observation.state": state,
                        "action": target_action,
                        "obj_init": np.asarray(env.obj_init_pose, dtype=np.float32),
                    },
                    task=env.instruction,
                )
            env.render(teleop=True, idx=episode_id)
    finally:
        env.env.close_viewer()
        shutil.rmtree(dataset.root / "images", ignore_errors=True)
    print(f"采集完成：{episode_id}/{NUM_DEMOS}")
    return dataset


if RUN_INTERACTIVE_COLLECTION:
    dataset = collect_demonstrations()
else:
    print("采集默认关闭。确认 DISPLAY、路径和覆盖开关后，将 RUN_INTERACTIVE_COLLECTION 改为 True。")


键位与上游教程一致：`W/A/S/D` 控制平面移动，`R/F` 控制高度，`Q/E` 与方向键控制姿态，空格切换夹爪，`Z` 丢弃当前失败回合。每次成功保存后会重新关闭记录开关，避免把 reset 过程混进下一条 episode。


## Checkpoint 6：采集后审计


In [ ]:
info_path = COLLECTION_ROOT / "meta" / "info.json"
if info_path.exists():
    info, tasks = dataset_report(COLLECTION_ROOT)
    episodes_path = COLLECTION_ROOT / "meta" / "episodes.jsonl"
    episodes = [
        json.loads(line)
        for line in episodes_path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    task_counts = {}
    for episode in episodes:
        task = episode.get("tasks", [""])[0]
        task_counts[task] = task_counts.get(task, 0) + 1
    md_table(["指令", "episode 数"], sorted(task_counts.items()))
else:
    print("还没有可审计的数据：", COLLECTION_ROOT)


数据表通过后，再打开上游 `6.visualize_data.ipynb` 随机回放若干 episode。至少检查图像、state/action shape、夹爪开闭时序、是否真的抬起，以及红蓝杯指令是否平衡。完成这些检查后再进入 08–10 的训练 Notebook。
